# mRMR-QFS rework — Colab Pro scaling experiments (GPU)

Runs the two experiments that need more compute than a laptop:

1. **Scaling ablation** (n = 16,18,20,22): exact optimum vs strong classical baselines vs VQC(no-ent / ring / all) vs QAOA-mRMR. Confirms at larger n that no quantum variant beats a trivial classical mean-field optimiser.
2. **Decoder-criterion validation grid**: across architectures × noise levels, check that the closed-form critical rate (Cor.4) and sampling bound (Prop.5) predict the expval→sampling crossover — this is the paper's headline predictive protocol.

**Runtime menu (top): set Runtime → Change runtime type → GPU.** The code uses `lightning.gpu` if available, else `lightning.qubit`, else `default.qubit`.

---

### How to run (Google Drive + resumable)
1. Upload the whole **`mRMR_rework`** folder to your Google Drive (anywhere under *My Drive*).
2. Open this notebook from inside that folder and run all cells. The second cell mounts Drive and auto-locates the folder.
3. **Crash-safe / resumable:** every result is written to `mRMR_rework/results/` on Drive, and each experiment checkpoints into `mRMR_rework/checkpoints/` as it goes. If Colab disconnects or times out, just **re-run the same cell** — finished `(n,lam,seed)` combos (Exp 1) and finished seeds (Exp 2) are skipped, and it picks up from where it stopped. To force a clean rerun, delete the matching folder under `checkpoints/`.

In [ ]:
!pip -q install pennylane scikit-learn >/dev/null 2>&1
# GPU plugin (optional; ignore failure and fall back to CPU lightning)
!pip -q install pennylane-lightning-gpu custatevec-cu12 >/dev/null 2>&1 || echo 'no gpu plugin, using CPU'
import pennylane as qml, numpy as np, time, json
for name in ['lightning.gpu','lightning.qubit','default.qubit']:
    try:
        qml.device(name, wires=2); DEV=name; break
    except Exception: pass
DIFF = 'adjoint' if DEV!='default.qubit' else 'backprop'
print('device =', DEV, '/', DIFF, '| pennylane', qml.__version__)

In [ ]:
# === Mount Google Drive + persistent results / resumable checkpoints ===
# Upload the whole `mRMR_rework` folder to your Drive, open this notebook from
# inside it, and run. Results + checkpoints are written back into that folder so
# they survive disconnects and let every experiment resume where it stopped.
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json

def _find_drive_base():
    # auto-locate the uploaded mRMR_rework folder (search <=2 levels under MyDrive)
    for pat in ('/content/drive/MyDrive/mRMR_rework',
                '/content/drive/MyDrive/*/mRMR_rework',
                '/content/drive/MyDrive/*/*/mRMR_rework'):
        hits = [c for c in glob.glob(pat) if os.path.isdir(c)]
        if hits:
            return sorted(hits, key=lambda x: x.count('/'))[0]
    return '/content/drive/MyDrive/mRMR_rework'  # default if not found

DRIVE_BASE  = _find_drive_base()        # <-- if auto-detect is wrong, set this manually
RESULTS_DIR = f'{DRIVE_BASE}/results'
CKPT_DIR    = f'{DRIVE_BASE}/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

def save_json(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)   # atomic write: a crash mid-save can't corrupt a checkpoint
def load_json(path):
    with open(path) as f:
        return json.load(f)

print('DRIVE_BASE  =', DRIVE_BASE)
print('RESULTS_DIR =', RESULTS_DIR)
print('CKPT_DIR    =', CKPT_DIR)

## Engine (writes qfs_common.py — identical to the local repo)

In [ ]:
%%writefile qfs_common.py
import os
for _v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v,'1')
import itertools, numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
def make_hard_instance(n_A=6,n_B=10,n_C=6,n_samples=800,rho_A=0.92,w_A=2.2,w_B=1.0,d_B=6,noise_B=0.35,label_noise=0.5,seed=0):
    rng=np.random.default_rng(seed); N=n_samples
    s_A=rng.standard_normal(N); s_B=rng.standard_normal((n_B,N))
    logit=w_A*s_A+w_B*s_B[:d_B].sum(0); logit=(logit-logit.mean())/(logit.std()+1e-9)
    y=(logit+label_noise*rng.standard_normal(N)>0).astype(int)
    cols,groups=[],[]
    for _ in range(n_A): cols.append(rho_A*s_A+np.sqrt(max(1-rho_A**2,0))*rng.standard_normal(N)); groups.append('A')
    for j in range(n_B): cols.append(np.sqrt(max(1-noise_B**2,0))*s_B[j]+noise_B*rng.standard_normal(N)); groups.append('B' if j<d_B else 'b')
    for _ in range(n_C): cols.append(rng.standard_normal(N)); groups.append('C')
    return np.column_stack(cols), y, np.array(groups)
def _discretize(X,n_bins=8):
    Xb=np.zeros_like(X,dtype=int)
    for i in range(X.shape[1]):
        e=np.quantile(X[:,i],np.linspace(0,1,n_bins+1)[1:-1]); Xb[:,i]=np.digitize(X[:,i],e)
    return Xb
def _mi(a,b):
    Na,Nb=int(a.max())+1,int(b.max())+1; j=np.zeros((Na,Nb)); np.add.at(j,(a,b),1.0); j/=j.sum()
    pa=j.sum(1,keepdims=True); pb=j.sum(0,keepdims=True); nz=j>0
    return float(np.sum(j[nz]*np.log(j[nz]/(pa@pb)[nz])))
def build_qubo(X,y,n_bins=8):
    Xb=_discretize(X,n_bins); n=X.shape[1]
    r=np.array([_mi(Xb[:,i],y) for i in range(n)]); R=np.zeros((n,n))
    for i in range(n):
        for jj in range(i+1,n): R[i,jj]=R[jj,i]=_mi(Xb[:,i],Xb[:,jj])
    r=r/(r.max()+1e-12); R=R/(R.max()+1e-12); np.fill_diagonal(R,0.0); return r,R
def eval_J(S,r,R,lam): S=list(S); return float(r[S].sum()-lam*R[np.ix_(S,S)].sum()/2.0)
def exact_solve(r,R,k,lam):
    n=len(r); flat=np.fromiter(itertools.chain.from_iterable(itertools.combinations(range(n),k)),dtype=np.int64)
    combos=flat.reshape(-1,k); J=r[combos].sum(1)
    for a in range(k):
        for b in range(a+1,k): J=J-lam*R[combos[:,a],combos[:,b]]
    return J,combos,int(np.argmax(J))
def relevance_topk(r,k): return np.argsort(-r)[:k]
def random_mrmr(r,R,k,lam,shots=4096,seed=0):
    rng=np.random.default_rng(seed); n=len(r); bJ,bS=-np.inf,None
    for _ in range(shots):
        S=rng.choice(n,k,replace=False); v=eval_J(S,r,R,lam)
        if v>bJ: bJ,bS=v,S
    return np.array(sorted(bS))
def greedy_mrmr(r,R,k,lam):
    n=len(r); S=[]
    for _ in range(k):
        best,bi=-np.inf,None
        for i in range(n):
            if i in S: continue
            gn=r[i]-lam*(R[i,S].sum() if S else 0.0)
            if gn>best: best,bi=gn,i
        S.append(bi)
    return np.array(sorted(S))
def simulated_annealing(r,R,k,lam,iters=15000,T0=0.25,seed=0):
    rng=np.random.default_rng(seed); n=len(r); S=set(rng.choice(n,k,replace=False).tolist())
    cur=eval_J(list(S),r,R,lam); best,bS=cur,set(S)
    for t in range(iters):
        T=T0*(1-t/iters)+1e-4; out=[o for o in range(n) if o not in S]
        i=rng.choice(list(S)); jj=rng.choice(out); S2=set(S); S2.discard(i); S2.add(jj)
        nv=eval_J(list(S2),r,R,lam)
        if nv>cur or rng.random()<np.exp((nv-cur)/T):
            S,cur=S2,nv
            if nv>best: best,bS=nv,set(S)
    return np.array(sorted(bS))
def classical_meanfield(r,R,k,lam,T=500,lr=0.05,alpha=5.0,tau0=1.0,tau1=10.0,restarts=8,seed=0):
    rng=np.random.default_rng(seed); n=len(r); bJ,bS=-np.inf,None
    for _ in range(restarts):
        th=rng.normal(0,0.1,n); m=np.zeros(n); v=np.zeros(n)
        for t in range(T):
            tau=tau0+(tau1-tau0)*t/max(T-1,1); p=1/(1+np.exp(-tau*th))
            dg=r-lam*(R@p)-2*alpha*(p.sum()-k); grad=dg*(tau*p*(1-p)); g=-grad
            m=0.9*m+0.1*g; v=0.999*v+0.001*g*g; mh=m/(1-0.9**(t+1)); vh=v/(1-0.999**(t+1))
            th-=lr*mh/(np.sqrt(vh)+1e-8)
        p=1/(1+np.exp(-tau1*th)); S=np.argsort(-p)[:k]; vJ=eval_J(S,r,R,lam)
        if vJ>bJ: bJ,bS=vJ,S
    return np.array(sorted(bS))
def downstream_svm(X,y,S,seed=0):
    Xs=StandardScaler().fit_transform(X[:,list(S)]); cv=StratifiedKFold(5,shuffle=True,random_state=seed)
    return float(cross_val_score(SVC(kernel='rbf'),Xs,y,cv=cv).mean())

## Experiment 1 — scaling ablation (n = 16,18,20,22)

In [ ]:
import qfs_common as q
from pennylane import numpy as pnp
NS=[16,18,20,22]; LAMS=[1.0,2.0]; SEEDS=list(range(8))
VQC=dict(L=4,steps=240,restarts=3,lr=0.06,tau1=10.0,alpha=5.0)
SCALING_CKPT=f'{CKPT_DIR}/scaling'; os.makedirs(SCALING_CKPT,exist_ok=True)
def groups(n):
    nA=max(2,round(n*0.25)); nC=max(2,round(n*0.22)); nB=n-nA-nC; return nA,nB,nC,max(3,round(nB*0.7))
def qnode(n,L,ent):
    dev=qml.device(DEV,wires=n)
    def circ(theta,r):
        for i in range(n): qml.RY(r[i]*np.pi,wires=i)
        for l in range(L):
            for i in range(n): qml.RY(theta[l,i,0],wires=i); qml.RZ(theta[l,i,1],wires=i)
            if ent=='ring':
                for i in range(n): qml.CNOT(wires=[i,(i+1)%n])
            elif ent=='all':
                for i in range(n):
                    for jj in range(i+1,n): qml.CNOT(wires=[i,jj])
            if l<L-1:
                for i in range(n): qml.RY(r[i]*np.pi/2,wires=i)
        return [qml.expval(qml.PauliZ(i)) for i in range(n)]
    return qml.QNode(circ,dev,diff_method=DIFF,interface='autograd')
def vqc_select(r,R,k,lam,ent,seed):
    n=len(r); L=VQC['L']; qn=qnode(n,L,ent)
    rp=pnp.array(r,requires_grad=False); Rp=pnp.array(R,requires_grad=False); bl,bS=np.inf,None
    for rs in range(VQC['restarts']):
        rng=np.random.default_rng(1000*seed+rs); th=pnp.array(rng.uniform(-np.pi,np.pi,(L,n,2)),requires_grad=True)
        opt=qml.AdamOptimizer(VQC['lr']); fin=None
        for t in range(VQC['steps']):
            tau=1.0+(VQC['tau1']-1.0)*t/max(VQC['steps']-1,1)
            def cost(T):
                z=pnp.stack(qn(T,rp)); P=1/(1+pnp.exp(tau*z))
                return -pnp.sum(rp*P)+lam*0.5*pnp.sum(Rp*pnp.outer(P,P))+VQC['alpha']*(pnp.sum(P)-k)**2
            th,fin=opt.step_and_cost(cost,th)
        z=np.array(qn(th,rp)); P=1/(1+np.exp(VQC['tau1']*z)); S=np.array(sorted(np.argsort(-P)[:k]))
        if float(fin)<bl: bl,bS=float(fin),S
    return bS
# --- resumable loop: one checkpoint per (n, lam, seed); reruns skip finished combos ---
combos=[(n,lam,sd) for n in NS for lam in LAMS for sd in SEEDS]; ntotal=len(combos)
ck_path=lambda n,lam,sd: f'{SCALING_CKPT}/n{n}_lam{lam}_sd{sd}.json'
done=[c for c in combos if os.path.exists(ck_path(*c))]
_secs=[load_json(ck_path(*c)).get('secs',0) for c in done]           # times from finished combos
_avg=float(np.mean([s for s in _secs if s>0])) if any(s>0 for s in _secs) else 0.0
ndone=len(done)
msg=f'scaling ablation: {ndone}/{ntotal} combos already done'
if _avg: msg+=f' | avg {_avg:.0f}s/combo -> ETA ~{_avg*(ntotal-ndone)/60:.0f} min for the rest'
print(msg+' -> resuming',flush=True)
t0=time.time()
for (n,lam,sd) in combos:
    ck=ck_path(n,lam,sd)
    if os.path.exists(ck): continue
    tc=time.time()
    nA,nB,nC,dB=groups(n); k=max(2,round(0.42*n))
    X,y,_=q.make_hard_instance(n_A=nA,n_B=nB,n_C=nC,d_B=dB,n_samples=600,seed=sd)
    r,R=q.build_qubo(X,y); Ja,co,bi=q.exact_solve(r,R,k,lam); Jx=float(Ja[bi]); ex=q.downstream_svm(X,y,co[bi],seed=sd)
    sel={'Random':q.random_mrmr(r,R,k,lam,seed=sd),'Greedy':q.greedy_mrmr(r,R,k,lam),
         'SA':q.simulated_annealing(r,R,k,lam,seed=sd),'MeanField':q.classical_meanfield(r,R,k,lam,seed=sd),
         'VQC(no-ent)':vqc_select(r,R,k,lam,'none',sd),'VQC(ring)':vqc_select(r,R,k,lam,'ring',sd),
         'VQC(all)':vqc_select(r,R,k,lam,'all',sd)}
    rows_c=[dict(n=n,k=k,lam=lam,seed=sd,method=m,dJ=Jx-q.eval_J(S,r,R,lam),dSVM=ex-q.downstream_svm(X,y,S,seed=sd)) for m,S in sel.items()]
    secs=time.time()-tc
    save_json(ck,{'combo':[n,lam,sd],'secs':round(secs,1),'rows':rows_c})
    ndone+=1
    print(f'[{ndone}/{ntotal}] n={n} lam={lam} sd={sd}  {secs:.0f}s  (session elapsed {(time.time()-t0)/60:.1f} min)',flush=True)
# aggregate every per-combo checkpoint into the final results file (tolerates old list-format ckpts)
rows=[]
for c in combos:
    if os.path.exists(ck_path(*c)):
        d=load_json(ck_path(*c)); rows.extend(d['rows'] if isinstance(d,dict) else d)
save_json(f'{RESULTS_DIR}/scaling_ablation.json',rows)
print(f'saved {RESULTS_DIR}/scaling_ablation.json  ({len(rows)} rows)')

In [ ]:
import pandas as pd
try: rows
except NameError: rows=load_json(f'{RESULTS_DIR}/scaling_ablation.json')  # reload after a fresh session
df=pd.DataFrame(rows)
summ=df.groupby(['n','method']).agg(dJ=('dJ','mean'),dJ_pos=('dJ',lambda s:(s>1e-6).mean()),dSVM=('dSVM','mean')).round(3)
print(summ.to_string())
print('\nKEY: every quantum row dJ>=0 and >= MeanField  =>  no quantum variant beats trivial classical, at all n.')

In [ ]:
# === Experiment 1b — QAOA-mRMR at n=16..22 (fills the QAOA-mRMR row of tab:scaling) ===
# Run AFTER cells 1, 2, 4 (device + Drive + engine). Uses the SAME instances, seeds,
# k, and lambda as Exp 1, so the QAOA dJ is directly comparable to the VQC rows.
# QAOA config (p=3, steps=150, lr=0.04, mu=2.0) is IDENTICAL to phase2_ablation_strong.py
# (the n=12,14 source of tab:ablation); the only change is that the cost energy is
# evaluated as one Hamiltonian expectation (adjoint-friendly, so n=22 is tractable).
# Resumable: one checkpoint per (n,lam,seed) under checkpoints/qaoa/ -> re-run to resume.
import qfs_common as q
import os, time, numpy as np
import pennylane as qml
from pennylane import numpy as pnp

NS=[16,18,20,22]; LAMS=[1.0,2.0]; SEEDS=list(range(8))
QAOA=dict(p=3, steps=150, lr=0.04, mu=2.0)
QAOA_CKPT=f'{CKPT_DIR}/qaoa'; os.makedirs(QAOA_CKPT, exist_ok=True)
def groups(n):
    nA=max(2,round(n*0.25)); nC=max(2,round(n*0.22)); nB=n-nA-nC; return nA,nB,nC,max(3,round(nB*0.7))

def qaoa_mrmr_select(r,R,k,lam,seed,cfg):
    n=len(r); p=cfg['p']; mu=cfg['mu']
    # QUBO f(x) = -sum r x + lam sum_{i<j} R x x + mu (sum x - k)^2 ; x=(1-Z)/2 -> Ising h,J
    Q=np.zeros((n,n))
    for i in range(n): Q[i,i]=-r[i]+mu*(1-2*k)
    for i in range(n):
        for j in range(i+1,n): Q[i,j]=lam*R[i,j]+2*mu
    h=np.zeros(n); Jd={}
    for i in range(n):
        h[i]+=-Q[i,i]/2.0
        for j in range(i+1,n):
            Jd[(i,j)]=Q[i,j]/4.0; h[i]+=-Q[i,j]/4.0; h[j]+=-Q[i,j]/4.0
    coeffs=[float(h[i]) for i in range(n)]+[float(Jd[(i,j)]) for (i,j) in Jd]
    ops=[qml.PauliZ(i) for i in range(n)]+[qml.PauliZ(i)@qml.PauliZ(j) for (i,j) in Jd]
    Hcost=qml.Hamiltonian(coeffs, ops)            # energy QAOA minimises (== phase2's sum h<Z>+J<ZZ>)
    dev=qml.device(DEV, wires=n)
    def _ansatz(g,b):
        for i in range(n): qml.Hadamard(wires=i)
        for layer in range(p):
            for i in range(n):
                if abs(h[i])>1e-12: qml.RZ(2*g[layer]*h[i], wires=i)
            for (i,j),Jij in Jd.items():
                if abs(Jij)>1e-12:
                    qml.CNOT(wires=[i,j]); qml.RZ(2*g[layer]*Jij, wires=j); qml.CNOT(wires=[i,j])
            for i in range(n): qml.RX(2*b[layer], wires=i)
    def circ_energy(g,b): _ansatz(g,b); return qml.expval(Hcost)
    def circ_probs(g,b):  _ansatz(g,b); return qml.probs(wires=range(n))
    qn=qml.QNode(circ_energy, dev, diff_method=DIFF, interface='autograd')
    qn_probs=qml.QNode(circ_probs, dev, diff_method=None)
    rng=np.random.default_rng(seed)
    g=pnp.array(rng.uniform(0,np.pi,p), requires_grad=True)
    b=pnp.array(rng.uniform(0,np.pi,p), requires_grad=True)
    opt=qml.AdamOptimizer(cfg['lr'])
    def energy(gg,bb): return qn(gg,bb)
    for t in range(cfg['steps']):
        (g,b),_=opt.step_and_cost(energy, g, b)
    # decode: sample from probs, post-select |S|=k, rank by the true mRMR J (same as phase2)
    probs=np.array(qn_probs(g,b)); rngs=np.random.default_rng(seed+7)
    idx=rngs.choice(len(probs),4096,p=probs/probs.sum())
    bestJ,bestS=-np.inf,None
    for v in np.unique(idx):
        S=[i for i in range(n) if (v>>(n-1-i))&1]
        if len(S)==k:
            Jv=q.eval_J(S,r,R,lam)
            if Jv>bestJ: bestJ,bestS=Jv,sorted(S)
    if bestS is None:
        marg=np.zeros(n)
        for v,c in zip(*np.unique(idx,return_counts=True)):
            for i in range(n):
                if (v>>(n-1-i))&1: marg[i]+=c
        bestS=sorted(np.argsort(-marg)[:k])
    return np.array(bestS)

# --- resumable loop: one checkpoint per (n,lam,seed) ---
combos=[(n,lam,sd) for n in NS for lam in LAMS for sd in SEEDS]; ntot=len(combos)
ckp=lambda n,lam,sd: f'{QAOA_CKPT}/n{n}_lam{lam}_sd{sd}.json'
done=[c for c in combos if os.path.exists(ckp(*c))]
_secs=[load_json(ckp(*c)).get('secs',0) for c in done]
_avg=float(np.mean([s for s in _secs if s>0])) if any(s>0 for s in _secs) else 0.0
nd=len(done); msg=f'QAOA scaling: {nd}/{ntot} combos already done'
if _avg: msg+=f' | avg {_avg:.0f}s/combo -> ETA ~{_avg*(ntot-nd)/60:.0f} min for the rest'
print(msg+' -> resuming', flush=True)
t0=time.time()
for (n,lam,sd) in combos:
    ck=ckp(n,lam,sd)
    if os.path.exists(ck): continue
    tc=time.time()
    nA,nB,nC,dB=groups(n); k=max(2,round(0.42*n))
    X,y,_=q.make_hard_instance(n_A=nA,n_B=nB,n_C=nC,d_B=dB,n_samples=600,seed=sd)
    r,R=q.build_qubo(X,y); Ja,co,bi=q.exact_solve(r,R,k,lam); Jx=float(Ja[bi]); ex=q.downstream_svm(X,y,co[bi],seed=sd)
    S=qaoa_mrmr_select(r,R,k,lam,sd,QAOA)
    row=dict(n=n,k=k,lam=lam,seed=sd,method='QAOA-mRMR',
             dJ=Jx-q.eval_J(S,r,R,lam), dSVM=ex-q.downstream_svm(X,y,S,seed=sd))
    secs=time.time()-tc; save_json(ck,{'combo':[n,lam,sd],'secs':round(secs,1),'rows':[row]})
    nd+=1; print(f'[{nd}/{ntot}] n={n} lam={lam} sd={sd}  dJ={row["dJ"]:.3f}  {secs:.0f}s  (elapsed {(time.time()-t0)/60:.1f} min)', flush=True)
# aggregate -> results/scaling_qaoa.json
rows=[]
for c in combos:
    if os.path.exists(ckp(*c)): rows.extend(load_json(ckp(*c))['rows'])
save_json(f'{RESULTS_DIR}/scaling_qaoa.json', rows)
import collections
agg=collections.defaultdict(list)
for rw in rows: agg[rw['n']].append(rw['dJ'])
print(f'\nsaved {RESULTS_DIR}/scaling_qaoa.json  ({len(rows)} rows)')
print(f"{'n':>4}{'QAOA dJ mean':>16}")
for n in NS:
    if agg[n]: print(f'{n:>4}{np.mean(agg[n]):>16.3f}')
print('\nExpect: QAOA dJ > 0 and >= classical (~0), comparable to/above VQC(ring); fills tab:scaling.')

## Experiment 2 — decoder-criterion validation grid
Train VQC-ring noiselessly, evaluate under depolarizing noise across architectures, and check the expval→sampling crossover against the closed-form critical rate.

In [ ]:
# n=10 density-matrix is feasible; this validates Cor.4 / Prop.5 crossover prediction.
from pennylane import numpy as pnp
NQ=10; K=3; L=4; SEEDS2=list(range(5)); LEVELS=[0,0.001,0.002,0.005,0.01,0.02,0.05]
DECODER_CKPT=f'{CKPT_DIR}/decoder'; os.makedirs(DECODER_CKPT,exist_ok=True)
def train_ring(r,k,seed):
    dev=qml.device('default.qubit',wires=NQ)
    def circ(th,rr):
        for i in range(NQ): qml.RY(rr[i]*np.pi,wires=i)
        for l in range(L):
            for i in range(NQ): qml.RY(th[l,i,0],wires=i); qml.RZ(th[l,i,1],wires=i)
            for i in range(NQ): qml.CNOT(wires=[i,(i+1)%NQ])
            if l<L-1:
                for i in range(NQ): qml.RY(rr[i]*np.pi/2,wires=i)
        return [qml.expval(qml.PauliZ(i)) for i in range(NQ)]
    qn=qml.QNode(circ,dev,diff_method='backprop',interface='autograd')
    rng=np.random.default_rng(seed); th=pnp.array(rng.uniform(-np.pi,np.pi,(L,NQ,2)),requires_grad=True)
    rp=pnp.array(r,requires_grad=False); opt=qml.AdamOptimizer(0.06)
    for t in range(200):
        tau=1.0+9.0*t/199
        def cost(T):
            z=pnp.stack(qn(T,rp)); P=1/(1+pnp.exp(tau*z)); return -pnp.sum(rp*P)+5.0*(pnp.sum(P)-k)**2
        th=opt.step(cost,th)
    return np.array(th)
def noisy(th,r,s):
    dev=qml.device('default.mixed',wires=NQ); p1,p2=s,min(10*s,0.75)
    @qml.qnode(dev)
    def zc(T,rr):
        for i in range(NQ):
            qml.RY(rr[i]*np.pi,wires=i)
            if s>0: qml.DepolarizingChannel(p1,wires=i)
        for l in range(L):
            for i in range(NQ):
                qml.RY(T[l,i,0],wires=i); qml.RZ(T[l,i,1],wires=i)
                if s>0: qml.DepolarizingChannel(p1,wires=i)
            for i in range(NQ):
                jj=(i+1)%NQ; qml.CNOT(wires=[i,jj])
                if s>0: qml.DepolarizingChannel(p2,wires=i); qml.DepolarizingChannel(p2,wires=jj)
            if l<L-1:
                for i in range(NQ): qml.RY(rr[i]*np.pi/2,wires=i)
        return [qml.expval(qml.PauliZ(i)) for i in range(NQ)]
    @qml.qnode(dev)
    def pc(T,rr):
        for i in range(NQ):
            qml.RY(rr[i]*np.pi,wires=i)
            if s>0: qml.DepolarizingChannel(p1,wires=i)
        for l in range(L):
            for i in range(NQ):
                qml.RY(T[l,i,0],wires=i); qml.RZ(T[l,i,1],wires=i)
                if s>0: qml.DepolarizingChannel(p1,wires=i)
            for i in range(NQ):
                jj=(i+1)%NQ; qml.CNOT(wires=[i,jj])
                if s>0: qml.DepolarizingChannel(p2,wires=i); qml.DepolarizingChannel(p2,wires=jj)
            if l<L-1:
                for i in range(NQ): qml.RY(rr[i]*np.pi/2,wires=i)
        return qml.probs(wires=range(NQ))
    return np.array(zc(pnp.array(th,requires_grad=False),pnp.array(r,requires_grad=False))), \
           np.array(pc(pnp.array(th,requires_grad=False),pnp.array(r,requires_grad=False)))
G1,G2=3*L*NQ,L*NQ; p1star=np.log(10*0.7/np.log(9))/(G1+2*10*G2)
print('predicted expval critical p1* =',round(p1star,5))
# --- resumable loop: one checkpoint per seed; reruns skip finished seeds ---
seed_ck=lambda sd: f'{DECODER_CKPT}/seed{sd}.json'; nsd=len(SEEDS2)
done=[sd for sd in SEEDS2 if os.path.exists(seed_ck(sd))]
_secs=[load_json(seed_ck(sd)).get('secs',0) for sd in done]          # times from finished seeds
_avg=float(np.mean([s for s in _secs if s>0])) if any(s>0 for s in _secs) else 0.0
ndone=len(done)
msg=f'decoder grid: {ndone}/{nsd} seeds already done'
if _avg: msg+=f' | avg {_avg/60:.1f} min/seed -> ETA ~{_avg*(nsd-ndone)/60:.0f} min for the rest'
print(msg,flush=True)
t0=time.time()
for sd in SEEDS2:
    if os.path.exists(seed_ck(sd)):
        print('[resume] seed',sd,'already done',flush=True); continue
    tsd=time.time()
    X,y,_=q.make_hard_instance(n_A=2,n_B=NQ-4,n_C=2,d_B=3,n_samples=500,seed=sd); X=X[:,:NQ]
    r,R=q.build_qubo(X,y); th=train_ring(r,K,sd)
    sr={'expval':{},'topk':{},'sampling':{}}
    for s in LEVELS:
        z,pr=noisy(th,r,s)
        ev=sorted([i for i in range(NQ) if z[i]<0]); tk=sorted(np.argsort(z)[:K])
        rng=np.random.default_rng(sd); idx=rng.choice(len(pr),4096,p=np.clip(pr,0,None)/pr.sum())
        bJ,sp=-np.inf,None
        for v in np.unique(idx):
            S=[i for i in range(NQ) if (v>>(NQ-1-i))&1]
            if len(S)==K:
                Jv=q.eval_J(S,r,R,1.0)
                if Jv>bJ: bJ,sp=Jv,sorted(S)
        if sp is None: sp=tk
        sr['expval'][str(s)]=q.downstream_svm(X,y,ev,seed=sd) if ev else 0.5
        sr['topk'][str(s)]=q.downstream_svm(X,y,tk,seed=sd)
        sr['sampling'][str(s)]=q.downstream_svm(X,y,sp,seed=sd)
    sr['secs']=round(time.time()-tsd,1)
    save_json(seed_ck(sd),sr)
    ndone+=1
    print(f'[{ndone}/{nsd}] seed {sd} done  {sr["secs"]:.0f}s  (session elapsed {(time.time()-t0)/60:.1f} min)',flush=True)
# aggregate per-seed checkpoints back into the across-seed grid ('secs' key is ignored here)
res={d:{s:[] for s in LEVELS} for d in ['expval','topk','sampling']}
for sd in SEEDS2:
    if not os.path.exists(seed_ck(sd)): continue
    sr=load_json(seed_ck(sd))
    for d in res:
        for s in LEVELS: res[d][s].append(sr[d][str(s)])
print(f"\n{'s':>8}{'expval':>14}{'topk':>14}{'sampling':>14}")
for s in LEVELS:
    f=lambda d:f"{np.mean(res[d][s]):.3f}±{np.std(res[d][s]):.3f}"
    print(f"{s:>8}{f('expval'):>14}{f('topk'):>14}{f('sampling'):>14}")
save_json(f'{RESULTS_DIR}/decoder_grid.json',{d:{str(s):res[d][s] for s in LEVELS} for d in res})
print(f'\nsaved {RESULTS_DIR}/decoder_grid.json | expval should collapse near p1*, sampling stays high.')

## Results (saved to Drive)
Both experiments write their final files to **`mRMR_rework/results/`** on your Drive:
- `scaling_ablation.json`
- `decoder_grid.json`

These persist after the session ends — open them straight from Drive, no manual download needed. Per-run checkpoints live in `mRMR_rework/checkpoints/{scaling,decoder}/`; once the final JSONs look complete you can delete the `checkpoints/` folder to free space (or keep it to enable resume on a later rerun).